# 03 — Circuit Tracing

Interactive analysis of Phase 2 results.

Goals:
- Visualize attribution graphs for representative prompts
- Identify the divergence layer and confirm it is consistent across prompts
- Compare circuit structure across conditions (control vs. CoT unfaithful vs. sycophantic)
- Inspect consistent edges — the shared circuit backbone
- Validate ablation results: which features, when clamped, restore faithful behavior?

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

PHASE2_RUN = 'FILL_IN_RUN_ID'
PHASE2_DIR = Path('../experiments/results/phase2')

In [ ]:
# Load consistent edges for each condition
conditions = ['control', 'cot_unfaithful', 'sycophancy']
condition_edges = {}
for cond in conditions:
    p = PHASE2_DIR / PHASE2_RUN / cond / 'consistent_edges.json'
    if p.exists():
        with open(p) as f:
            condition_edges[cond] = json.load(f)
        print(f'{cond}: {len(condition_edges[cond])} consistent edges')
    else:
        print(f'Missing: {p}')

In [ ]:
# Edge overlap: which edges appear in both unfaithful conditions but not in control?
def edge_key(e):
    return (tuple(e['source']), tuple(e['target']))

if all(c in condition_edges for c in conditions):
    ctrl_keys = {edge_key(e) for e in condition_edges['control']}
    cot_keys = {edge_key(e) for e in condition_edges['cot_unfaithful']}
    syco_keys = {edge_key(e) for e in condition_edges['sycophancy']}
    
    override_specific = (cot_keys | syco_keys) - ctrl_keys
    shared_override = cot_keys & syco_keys - ctrl_keys
    
    print(f'Edges in CoT but not control: {len(cot_keys - ctrl_keys)}')
    print(f'Edges in sycophancy but not control: {len(syco_keys - ctrl_keys)}')
    print(f'Edges unique to override conditions (either): {len(override_specific)}')
    print(f'Edges shared between BOTH override conditions (not control): {len(shared_override)}')
    print()
    print('Shared override circuit edges (these are the candidates for the override mechanism):')
    for src, tgt in list(shared_override)[:10]:
        print(f'  Layer {src[0]}, Feature {src[1]} → Layer {tgt[0]}, Feature {tgt[1]}')

In [ ]:
# Load and visualize divergence layer analysis
div_path = PHASE2_DIR / PHASE2_RUN / 'divergence_layers.json'
if div_path.exists():
    with open(div_path) as f:
        div_data = json.load(f)
    
    layers = sorted(int(l) for l in div_data['divergence_frequency'].keys())
    freqs = [div_data['divergence_frequency'][str(l)] for l in layers]
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(layers, freqs, color='#b2182b', alpha=0.8)
    ax.axvline(div_data['peak_layer'], color='black', linestyle='--', linewidth=2,
               label=f'Peak divergence: layer {div_data["peak_layer"]}')
    ax.set_xlabel('Model layer')
    ax.set_ylabel('Fraction of prompts where this is the divergence layer')
    ax.set_title('Where does the model diverge from its internal representation?')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../paper/figures/divergence_layers.png', dpi=150)
    plt.show()
    
    print(f'Peak divergence layer: {div_data["peak_layer"]}')
    print(f'Fraction of prompts: {div_data["divergence_frequency"][str(div_data["peak_layer"])]:.2%}')

In [ ]:
# Display the pre-generated sample attribution graph
from IPython.display import Image

for cond in ['cot_unfaithful', 'sycophancy']:
    img_path = PHASE2_DIR / PHASE2_RUN / cond / 'sample_attribution_graph.png'
    if img_path.exists():
        print(f'--- {cond} ---')
        display(Image(str(img_path)))

In [ ]:
# Reconstruct a simple NetworkX graph from consistent edges for interactive exploration
def build_nx_from_edges(edges_list, min_freq=0.4):
    G = nx.DiGraph()
    for e in edges_list:
        if e['frequency'] >= min_freq:
            src = tuple(e['source'])
            tgt = tuple(e['target'])
            G.add_edge(src, tgt, weight=e['frequency'])
    return G

if 'cot_unfaithful' in condition_edges:
    G_cot = build_nx_from_edges(condition_edges['cot_unfaithful'], min_freq=0.4)
    print(f'CoT unfaithful circuit (freq>=0.4): {G_cot.number_of_nodes()} nodes, {G_cot.number_of_edges()} edges')
    
    # Nodes with highest in-degree are the convergence points of the override circuit
    top_convergence = sorted(G_cot.in_degree(), key=lambda x: x[1], reverse=True)[:5]
    print('Top convergence nodes (high in-degree):')
    for node, deg in top_convergence:
        print(f'  Layer {node[0]}, Feature {node[1]}: in-degree {deg}')